# Correlation Functions & Entanglement

**Correlation functions** from exact solution (Toeplitz determinants):
- $R^x_r = \langle\sigma^x_j \sigma^x_{j+r}\rangle$: nonzero only when $r \equiv 0 \pmod{3}$
- $R^y_r = \langle\sigma^y_j \sigma^y_{j+r}\rangle$: always nonvanishing

**Asymptotic behavior**:
- Cluster phase ($\lambda < 1$): $R^x_{3r}$ decays exponentially, $R^y_r \sim e^{-r/\xi}$
- Ising phase ($\lambda > 1$): $R^y_r \to (-1)^r m_y^2 > 0$, $R^x_{3r} \sim e^{-r/\xi}$
- Critical ($\lambda = 1$): algebraic decay

**Note**: The string order $O_z$ (cluster phase) is measured by the operator $\Omega_z(r) = \langle \sigma^x_i \sigma^y_{i+1} (\prod \sigma^z) \sigma^y_{j-1} \sigma^x_j \rangle$ [Eq. (47)], **not** by the bare $R^x$ correlation.

This notebook compares **exact**, **ED**, and **DMRG** correlation functions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.abspath('..'))

%matplotlib inline
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 6)})

## 0. Parameters

In [ ]:
# === System sizes ===
L_ed = 14           # ED system size
bc = 'open'

# === DMRG ===
run_dmrg = True
L_dmrg = 40         # 40 is reasonable (~1-2 min/point)
chi_max = 100

# === Lambda values to study ===
lam_list = [0.5, 1.0, 1.5]   # cluster, critical, Ising phases

# === Correlation distance ===
r_max_exact = 30    # for exact analytical

print(f"ED: L={L_ed}, bc={bc}")
print(f"DMRG: L={L_dmrg}, chi={chi_max}" if run_dmrg else "DMRG: skipped")
print(f"Lambda values: {lam_list}")

## 1. Exact correlation functions (thermodynamic limit)

In [ ]:
from cluster_ising.models.exact_solution import correlation_Rx, correlation_Ry

exact_corr = {}  # exact_corr[lam] = {'r': ..., 'Rx': ..., 'Ry': ...}

for lam in lam_list:
    r_vals = np.arange(1, r_max_exact + 1)
    Rx = np.array([correlation_Rx(r, lam) for r in r_vals])
    Ry = np.array([correlation_Ry(r, lam) for r in r_vals])
    exact_corr[lam] = {'r': r_vals, 'Rx': Rx, 'Ry': Ry}
    print(f"lam={lam}: R^x_3={Rx[2]:.6f}, R^y_1={Ry[0]:.6f}")

## 2. ED & DMRG correlation functions

In [ ]:
from cluster_ising.solvers.ed_solver import run_ed
from cluster_ising.observables.correlations import correlation_Rx as corr_Rx_num
from cluster_ising.observables.correlations import correlation_Ry as corr_Ry_num

ed_corr = {}  # ed_corr[lam] = {'r': ..., 'Rx': ..., 'Ry': ...}

for lam in lam_list:
    result = run_ed({'L': L_ed, 'lam': lam, 'bc': bc}, n_states=1)
    psi = result.state
    r_Rx, Rx = corr_Rx_num(psi, L_ed, state_type='vector')
    r_Ry, Ry = corr_Ry_num(psi, L_ed, state_type='vector')
    ed_corr[lam] = {'r_Rx': r_Rx, 'Rx': Rx, 'r_Ry': r_Ry, 'Ry': Ry}
    print(f"lam={lam}: ED done (L={L_ed})")

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

dmrg_corr = {}

if run_dmrg:
    from cluster_ising.solvers.dmrg_solver import run_dmrg as _run_dmrg
    from tqdm.auto import tqdm

    dmrg_params = {
        'trunc_params': {'chi_max': chi_max, 'svd_min': 1e-10},
        'mixer': True,
        'mixer_params': {'amplitude': 1e-5, 'decay': 1.5, 'disable_after': 30},
        'max_sweeps': 50,
        'max_E_err': 1e-10,
        'max_S_err': 1e-6,
    }

    for lam in tqdm(lam_list, desc='DMRG corr'):
        model_params = {
            'L': L_dmrg, 'lam': lam,
            'bc_MPS': 'finite', 'conserve': 'parity',
        }
        result = _run_dmrg(model_params, dmrg_params)
        psi = result.state
        r_Rx, Rx = corr_Rx_num(psi, L_dmrg, state_type='mps')
        r_Ry, Ry = corr_Ry_num(psi, L_dmrg, state_type='mps')
        dmrg_corr[lam] = {'r_Rx': r_Rx, 'Rx': Rx, 'r_Ry': r_Ry, 'Ry': Ry,
                          'psi': psi, 'energy': result.energy_per_site}
        print(f"lam={lam}: E/L={result.energy_per_site:.8f}, chi_max={max(psi.chi)}")
else:
    print("DMRG skipped")

## 3. Comparison Plots: $R^x_r$ and $R^y_r$

In [ ]:
fig, axes = plt.subplots(len(lam_list), 2, figsize=(14, 4 * len(lam_list)))
if len(lam_list) == 1:
    axes = axes.reshape(1, -1)

phase_names = {}
for lam in lam_list:
    if lam < 1:
        phase_names[lam] = 'Cluster'
    elif lam == 1:
        phase_names[lam] = 'Critical'
    else:
        phase_names[lam] = 'Ising'

for row, lam in enumerate(lam_list):
    # --- R^x ---
    ax = axes[row, 0]
    ec = exact_corr[lam]
    ax.plot(ec['r'], ec['Rx'], 'k.-', lw=1.5, ms=4, label='Exact', alpha=0.8)

    edc = ed_corr[lam]
    ax.plot(edc['r_Rx'], edc['Rx'], 'bo', ms=5, alpha=0.6, label=f'ED (L={L_ed})')

    if lam in dmrg_corr:
        dc = dmrg_corr[lam]
        ax.plot(dc['r_Rx'], dc['Rx'], 'r^', ms=4, alpha=0.6, label=f'DMRG (L={L_dmrg})')

    ax.set_xlabel('$r$')
    ax.set_ylabel('$R^x_r$')
    ax.set_title(f'$R^x_r$ at $\\lambda = {lam}$ ({phase_names[lam]} phase)')
    ax.legend(fontsize=8)
    ax.axhline(0, color='gray', ls='-', alpha=0.3)

    # --- R^y ---
    ax = axes[row, 1]
    ax.plot(ec['r'], ec['Ry'], 'k.-', lw=1.5, ms=4, label='Exact', alpha=0.8)
    ax.plot(edc['r_Ry'], edc['Ry'], 'bo', ms=5, alpha=0.6, label=f'ED (L={L_ed})')

    if lam in dmrg_corr:
        dc = dmrg_corr[lam]
        ax.plot(dc['r_Ry'], dc['Ry'], 'r^', ms=4, alpha=0.6, label=f'DMRG (L={L_dmrg})')

    ax.set_xlabel('$r$')
    ax.set_ylabel('$R^y_r$')
    ax.set_title(f'$R^y_r$ at $\\lambda = {lam}$ ({phase_names[lam]} phase)')
    ax.legend(fontsize=8)
    ax.axhline(0, color='gray', ls='-', alpha=0.3)

fig.suptitle('Correlation Functions: Exact vs ED vs DMRG', fontsize=14, y=1.01)
fig.tight_layout()
plt.savefig('correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Log-scale decay: correlation length extraction

In [ ]:
from cluster_ising.observables.correlations import correlation_length

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Rx for multiples of 3 only
ax = axes[0]
for lam in lam_list:
    ec = exact_corr[lam]
    mask3 = (ec['r'] % 3 == 0)
    r3 = ec['r'][mask3]
    Rx3 = np.abs(ec['Rx'][mask3])
    valid = Rx3 > 1e-15
    if np.any(valid):
        ax.semilogy(r3[valid], Rx3[valid], 'o-', ms=4,
                   label=f'$\\lambda = {lam}$')

ax.set_xlabel('$r$ (multiples of 3)')
ax.set_ylabel('$|R^x_r|$')
ax.set_title('$|R^x_r|$ decay (log scale)')
ax.legend()

# Ry
ax = axes[1]
for lam in lam_list:
    ec = exact_corr[lam]
    Ry_abs = np.abs(ec['Ry'])
    valid = Ry_abs > 1e-15
    ax.semilogy(ec['r'][valid], Ry_abs[valid], 'o-', ms=4,
               label=f'$\\lambda = {lam}$')
    # Extract correlation length
    xi = correlation_length(ec['r'], ec['Ry'])
    if np.isfinite(xi):
        print(f"lam={lam}: xi_y = {xi:.3f}")

ax.set_xlabel('$r$')
ax.set_ylabel('$|R^y_r|$')
ax.set_title('$|R^y_r|$ decay (log scale)')
ax.legend()

fig.tight_layout()
plt.show()

## 5. Temperature dependence of correlations

In [ ]:
lam_temp = 0.5  # cluster phase
betas = [np.inf, 10.0, 5.0, 2.0, 1.0]  # inverse temperatures
r_vals_t = np.arange(1, 20)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for beta in betas:
    label = '$T=0$' if np.isinf(beta) else f'$\\beta = {beta}$'
    Rx_t = np.array([correlation_Rx(r, lam_temp, beta) for r in r_vals_t])
    Ry_t = np.array([correlation_Ry(r, lam_temp, beta) for r in r_vals_t])
    axes[0].plot(r_vals_t, Rx_t, 'o-', ms=3, label=label)
    axes[1].plot(r_vals_t, Ry_t, 'o-', ms=3, label=label)

axes[0].set_xlabel('$r$'); axes[0].set_ylabel('$R^x_r$')
axes[0].set_title(f'$R^x_r(T)$ at $\\lambda = {lam_temp}$')
axes[0].legend(fontsize=9); axes[0].axhline(0, color='gray', ls='-', alpha=0.3)

axes[1].set_xlabel('$r$'); axes[1].set_ylabel('$R^y_r$')
axes[1].set_title(f'$R^y_r(T)$ at $\\lambda = {lam_temp}$')
axes[1].legend(fontsize=9); axes[1].axhline(0, color='gray', ls='-', alpha=0.3)

fig.suptitle('Finite-temperature correlation functions', fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## 6. Entanglement entropy profiles across the phase diagram

In [ ]:
from cluster_ising.observables.entanglement import block_entropies

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (a) S(l) profiles from ED at different lambda
ax = axes[0]
for lam in [0.3, 0.7, 1.0, 1.5, 2.5]:
    result = run_ed({'L': L_ed, 'lam': lam, 'bc': bc}, n_states=1)
    S = block_entropies(result.state, L_ed, 'vector')
    bonds = np.arange(1, L_ed)
    ax.plot(bonds, S, 'o-', ms=3, label=f'$\\lambda = {lam}$')

ax.set_xlabel('Bond $l$')
ax.set_ylabel('$S(l)$')
ax.set_title(f'Block entropy (ED, L={L_ed})')
ax.legend(fontsize=9)

# (b) Half-chain entropy vs lambda
ax = axes[1]
from cluster_ising.observables.entanglement import half_chain_entropy

lam_sweep = np.linspace(0.1, 3.0, 40)
S_half = []
for lam in lam_sweep:
    result = run_ed({'L': L_ed, 'lam': lam, 'bc': bc}, n_states=1)
    S_half.append(half_chain_entropy(result.state, L_ed, 'vector'))

ax.plot(lam_sweep, S_half, 'b.-', ms=4)
ax.axvline(1.0, color='red', ls='--', alpha=0.5, label='$\\lambda_c = 1$')
ax.set_xlabel('$\\lambda$')
ax.set_ylabel('$S_{L/2}$')
ax.set_title(f'Half-chain entropy vs $\\lambda$ (L={L_ed})')
ax.legend()

fig.tight_layout()
plt.savefig('entanglement_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. DMRG entanglement spectrum at critical point

In [ ]:
if run_dmrg and 1.0 in dmrg_corr:
    from cluster_ising.observables.entanglement import entanglement_spectrum, block_entropies

    psi_crit = dmrg_corr[1.0]['psi']

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # (a) S(l) profile from DMRG
    ax = axes[0]
    S_profile = block_entropies(psi_crit, L_dmrg, 'mps')
    bonds = np.arange(1, L_dmrg)
    ax.plot(bonds, S_profile, 'r-', lw=1.5)
    ax.set_xlabel('Bond $l$')
    ax.set_ylabel('$S(l)$')
    ax.set_title(f'Block entropy (DMRG, L={L_dmrg}, $\\lambda=1$)')

    # (b) Entanglement spectrum
    ax = axes[1]
    sv = entanglement_spectrum(psi_crit, state_type='mps')
    ax.semilogy(sv[:30], 'ro', ms=5)
    ax.set_xlabel('Index')
    ax.set_ylabel('Schmidt value')
    ax.set_title(f'Entanglement spectrum at center (L={L_dmrg}, $\\lambda=1$)')

    fig.tight_layout()
    plt.show()
else:
    print("DMRG at lambda=1.0 not available (run DMRG section first)")